# NVDA Intraday Volatility Dataset Construction

**Author:** Ayooluwa Adelagun

**Date:** March 2026

---

## Overview

This notebook constructs a supervised machine learning dataset for predicting the **direction of daily intraday volatility** for NVIDIA Corporation (NVDA).

Intraday volatility is estimated using the **Root Mean Square (RMS) of hourly log returns** within each trading day. The target variable is binary: `1` if today's intraday volatility is greater than yesterday's, `0` otherwise.

Here, we add **VIX** (the CBOE Volatility Index) as an additional feature. VIX measures the market's 30-day implied volatility expectation and is a well-documented predictor of individual stock volatility.

### Data Sources
- `NVDA_full_1hour_adjsplit.txt` — Split-adjusted hourly OHLCV data from 2000-01-03
- `NVDA_full_1day_adjsplit.txt` — Split-adjusted daily OHLCV data from 2000-01-04
- VIX daily closing prices downloaded via `yfinance`

### Pipeline Summary
1. Load and preprocess hourly and daily price data
2. Compute hourly log returns and aggregate to daily intraday volatility
3. Engineer feature variables from daily price data
4. Add VIX as a forward-looking implied volatility feature
5. Construct the binary target variable
6. Export the final dataset to CSV

### Feature Variables
| Feature | Description |
|---|---|
| `Return_Squared` | Squared daily log return (variance proxy) |
| `Return` | Daily log return |
| `Range` | Daily high minus low (price range) |
| `Volume` | Daily trading volume |
| `VIX` | CBOE Volatility Index closing price (30-day implied volatility) |
| `Daily Volatility` | RMS of hourly log returns (intraday volatility estimate) |
| `target` | 1 if volatility increased vs prior day, else 0 |

---
## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

---
## 2. Load & Preprocess Data

Both files have no header row, so column names are assigned manually.

The hourly dataset begins on `2000-01-03`, one day before the daily dataset starts on `2000-01-04`. The first 8 rows (all hourly bars for `2000-01-03`) are dropped to ensure alignment between the two datasets.

**Hourly log return** is computed as:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

where $P_t$ is the hourly closing price.

In [2]:
# Files have no header row so column names are assigned manually
data_1hour = pd.read_csv('NVDA_full_1hour_adjsplit.txt', header=None)
data_1hour.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

data_1day = pd.read_csv('NVDA_full_1day_adjsplit.txt', header=None)
data_1day.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

# Drops the first 8 rows which correspond to 2000-01-03. The hourly dataset starts 
# on 2000-01-03 09:00:00, while the daily dataset starts on 2000-01-04. This means that 
# the first day in the hourly dataset has no corresponding day in the daily dataset, so 
# we drop it to ensure both datasets are aligned for analysis.
data_1hour_dropped = data_1hour.drop(list(range(8)), axis=0)
data_1hour_dropped = pd.DataFrame(data_1hour_dropped)

# Convert Date column from string to datetime for time-based operations
data_1hour_dropped['Date'] = pd.to_datetime(data_1hour_dropped['Date'], format='%Y-%m-%d %H:%M:%S')

# Close must be explicitly cast to float before any numeric operations
data_1hour_dropped['Close'] = data_1hour_dropped['Close'].astype(float)

# Hourly log return: ln(P_t) - ln(P_{t-1})
# .diff() subtracts the previous row, giving the period-over-period log change
data_1hour_dropped['Hourly Log Return'] = np.log(data_1hour_dropped['Close']).diff()

print(data_1hour_dropped.head(10))

                  Date    Open    High     Low   Close    Volume  \
8  2000-01-04 09:00:00  0.0958  0.0961  0.0932  0.0939  46368000   
9  2000-01-04 10:00:00  0.0940  0.0952  0.0940  0.0952  68112000   
10 2000-01-04 11:00:00  0.0952  0.0961  0.0944  0.0951  18384000   
11 2000-01-04 12:00:00  0.0951  0.0961  0.0943  0.0943  30624000   
12 2000-01-04 13:00:00  0.0943  0.0945  0.0901  0.0928  49536000   
13 2000-01-04 14:00:00  0.0924  0.0939  0.0917  0.0937  16176000   
14 2000-01-04 15:00:00  0.0937  0.0949  0.0930  0.0949  60240000   
15 2000-01-04 16:00:00  0.0949  0.0949  0.0949  0.0949   1824000   
16 2000-01-05 09:00:00  0.0922  0.0937  0.0917  0.0922  29232000   
17 2000-01-05 10:00:00  0.0922  0.0930  0.0911  0.0917  95712000   

    Hourly Log Return  
8                 NaN  
9            0.013750  
10          -0.001051  
11          -0.008448  
12          -0.016035  
13           0.009652  
14           0.012726  
15           0.000000  
16          -0.028864  
17         

---
## 3. Compute Intraday Volatility

Intraday volatility for each trading day is estimated as the **Root Mean Square (RMS)** of the hourly log returns within that day:

$$\sigma_{\text{intraday}} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} r_i^2}$$

where $r_i$ are the hourly log returns and $N$ is the number of valid (non-null) observations that day.

Hourly data is resampled to daily frequency using `.resample('D')`. Weekend and holiday dates where no trading occurred will produce `NaN` or zero volatility and are removed in the next step.

In [3]:
def calculate_intraday_volatility(dataframe):
    # Working on a copy to avoid modifying the original dataframe
    dataframe = dataframe.copy()
    dataframe['Date'] = pd.to_datetime(dataframe['Date'], format='%Y-%m-%d %H:%M:%S')

    # Then isolate the two columns needed and set Date as index for resampling
    date_log_return_df = dataframe[['Date', 'Hourly Log Return']]
    date_log_return_df = date_log_return_df.set_index('Date')

    # Resample to daily frequency and compute RMS of hourly log returns:
    #   Step 1: square each return and sum, divide by count of non-null values (mean of squares)
    #   Step 2: take the square root to get the RMS — the intraday volatility estimate
    intraday_volatility = (
        date_log_return_df
        .resample('D')
        .apply(lambda x: (x ** 2).sum(axis=0) / x.notna().sum(axis=0))  # mean of squared returns
        .apply(lambda x: x ** 0.5)                                        # square root -> RMS
    )

    intraday_volatility.reset_index(inplace=True)
    intraday_volatility.columns = ['Date', 'Intraday_Volatility']
    return intraday_volatility

result_df = calculate_intraday_volatility(data_1hour_dropped)
print(result_df.head(10))

/var/folders/1q/12m2t3kx34d184nxq9fqf_100000gp/T/ipykernel_81550/2786222816.py:16: RuntimeWarning: invalid value encountered in scalar divide
  .apply(lambda x: (x ** 2).sum(axis=0) / x.notna().sum(axis=0))  # mean of squared returns


        Date  Intraday_Volatility
0 2000-01-04             0.010513
1 2000-01-05             0.011202
2 2000-01-06             0.017537
3 2000-01-07             0.010706
4 2000-01-08                  NaN
5 2000-01-09                  NaN
6 2000-01-10             0.018222
7 2000-01-11             0.009431
8 2000-01-12             0.006539
9 2000-01-13             0.016232


---
## 4. Remove Zero-Volatility Days

Days where the computed intraday volatility is exactly zero are removed. These correspond to non-trading days (weekends, public holidays) that appear in the resampled index but contain no valid hourly return data.

In [4]:
def remove_zero_volatility(dataframe):
    # Exclude days with zero volatility — these are non-trading days (weekends, holidays)
    # that appear in the resampled index but contain no valid hourly return data
    filtered_dataframe = dataframe[dataframe['Intraday_Volatility'] != 0]
    return filtered_dataframe

newresults = remove_zero_volatility(result_df)
print(f"Rows after removing zero volatility: {len(newresults)}")

Rows after removing zero volatility: 9555


---
## 5. Engineer Feature Variables

The intraday volatility series is joined onto the daily OHLCV dataframe by date index alignment. Three additional features are then derived from the daily price data:

- **Daily Volatility**: intraday RMS volatility joined from the hourly data
- **Range**: `High - Low` — a simple measure of intraday price movement
- **Return**: daily log return computed from consecutive closing prices
- **Return_Squared**: squared daily log return, a standard proxy for daily variance

Rows with any missing values are dropped, which removes the first row (no prior close for log return).

In [5]:
# Convert Date columns to datetime in both dataframes before setting as index,
# ensuring they share the same type for alignment when joining
data_1day['Date'] = pd.to_datetime(data_1day['Date'])
newresults = newresults.copy()
newresults['Date'] = pd.to_datetime(newresults['Date'])

# Set Date as index on both dataframes so pandas can align rows by date when joining
data_1day.set_index('Date', inplace=True)
newresults.set_index('Date', inplace=True)

# Cast OHLCV columns to float — read as strings due to missing header row
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    data_1day[col] = data_1day[col].astype(float)

# Join intraday volatility onto the daily dataframe by matching date index
data_1day['Daily Volatility'] = newresults['Intraday_Volatility']

# Price range: simple measure of intraday price movement
data_1day['Range'] = data_1day['High'] - data_1day['Low']

# Daily log return: .diff() computes ln(Close_t) - ln(Close_{t-1})
data_1day['Return'] = np.log(data_1day['Close']).diff()

# Squared return: standard proxy for daily variance
data_1day['Return_Squared'] = np.square(data_1day['Return'])

# Drop NaNs: removes first row (no prior close for log return) and any
# days where volatility could not be computed
data_1day = data_1day.dropna()

print(data_1day.head())

              Open    High     Low   Close       Volume  Daily Volatility  \
Date                                                                        
2000-01-05  0.0922  0.0937  0.0905  0.0918  188352000.0          0.011202   
2000-01-06  0.0918  0.0918  0.0823  0.0858  120480000.0          0.017537   
2000-01-07  0.0854  0.0882  0.0841  0.0872   71184000.0          0.010706   
2000-01-10  0.0875  0.0937  0.0859  0.0901  239856000.0          0.018222   
2000-01-11  0.0896  0.0906  0.0865  0.0865  148128000.0          0.009431   

             Range    Return  Return_Squared  
Date                                          
2000-01-05  0.0032 -0.033211        0.001103  
2000-01-06  0.0095 -0.067593        0.004569  
2000-01-07  0.0041  0.016185        0.000262  
2000-01-10  0.0078  0.032716        0.001070  
2000-01-11  0.0041 -0.040776        0.001663  


---
## 6. Add VIX

**VIX** (the CBOE Volatility Index) is downloaded from Yahoo Finance and added as a feature. VIX measures the market's expectation of S&P 500 30-day implied volatility and is extensively documented as a predictor of individual stock volatility. For NVDA in particular, which has a beta of approximately 1.6–2.0, periods of elevated market fear (high VIX) are associated with amplified price movements regardless of NVDA-specific factors. VIX is stationary by construction, mean-reverting around a long-run level, which means it behaves well under the rolling MAD outlier filter used downstream.

In [6]:
import yfinance as yf

# Dynamically derive date range from the dataset
start_date = str(data_1day.index.min().date())
end_date   = str(data_1day.index.max().date())

# Download VIX daily closing prices from Yahoo Finance
# VIX ticker on Yahoo Finance is '^VIX'
vix_raw = yf.download(tickers='^VIX', start=start_date, end=end_date, auto_adjust=True)

# Strip timezone from index to allow alignment with data_1day (which has no timezone)
vix_raw.index = pd.to_datetime(vix_raw.index).tz_localize(None)

# Extract closing price — this is the standard VIX level used in the literature
vix_series = vix_raw['Close'].copy()
vix_series.name = 'VIX'

# Join VIX onto the daily dataframe by matching date index
# Days where VIX is missing (e.g. US holidays where equity markets are closed
# but the VIX settlement differs) are forward-filled with the prior day's value
data_1day['VIX'] = vix_series
data_1day['VIX'] = data_1day['VIX'].ffill()

# Drop any remaining NaNs (rows before VIX history begins)
data = data_1day.dropna()
daily_volatility = data['Daily Volatility']

print(f'Dataset shape after adding VIX: {data.shape}')
print(f'VIX date range: {vix_series.index.min().date()} to {vix_series.index.max().date()}')
print(f'VIX NaNs remaining: {data["VIX"].isna().sum()}')
print(data[['Close', 'VIX', 'Daily Volatility']].head(10))
print(f'\nVIX summary statistics:')
print(data['VIX'].describe().round(2))

[*********************100%***********************]  1 of 1 completed

Dataset shape after adding VIX: (6577, 10)
VIX date range: 2000-01-05 to 2026-02-27
VIX NaNs remaining: 0
             Close        VIX  Daily Volatility
Date                                           
2000-01-05  0.0918  26.410000          0.011202
2000-01-06  0.0858  25.730000          0.017537
2000-01-07  0.0872  21.719999          0.010706
2000-01-10  0.0901  21.709999          0.018222
2000-01-11  0.0865  22.500000          0.009431
2000-01-12  0.0842  22.840000          0.006539
2000-01-13  0.0878  21.709999          0.016232
2000-01-14  0.0915  19.660000          0.010579
2000-01-18  0.0954  21.500000          0.023247
2000-01-19  0.0945  21.719999          0.007560

VIX summary statistics:
count    6577.00
mean       19.82
std         8.35
min         9.14
25%        14.01
50%        17.77
75%        23.17
max        82.69
Name: VIX, dtype: float64


---
## 7. Construct Target Variable

The target variable is a binary classification label indicating whether intraday volatility **increased** relative to the prior trading day:

$$\text{target}_t = \begin{cases} 1 & \text{if } \sigma_t > \sigma_{t-1} \\ 0 & \text{otherwise} \end{cases}$$

The resulting dataset is approximately balanced between the two classes, which is desirable for classification.

In [7]:
# Build a temporary dataframe to construct the target without risk of index misalignment
df = pd.DataFrame({'daily_volatility': daily_volatility})

# Target = 1 if today's volatility exceeds yesterday's, else 0
# .shift(1) shifts the series down by one row so each value is compared to the prior day
df['target'] = np.where(df['daily_volatility'] > df['daily_volatility'].shift(1), 1, 0)

# First row has no prior day, so fill NaN with 0
df['target'] = df['target'].fillna(0)

# Join target back onto the main dataframe by date index
data['target'] = df['target']
data_full = data.dropna()

print(f"Final dataset shape: {data_full.shape}")
print(data_full['target'].value_counts())

Final dataset shape: (6577, 11)
target
0    3293
1    3284
Name: count, dtype: int64


---
## 8. Finalise Dataset

The dataset is trimmed to start from **2005-01-03** to match the intrahour dataset date range and remove the structurally different high-volatility regime observed in NVDA's early years (2000–2004). This ensures both datasets are directly comparable over the same time period.

Columns are then reordered to match the canonical feature ordering used in the modelling pipeline.

In [ ]:
# Filter to start from 2005-01-03 to match the intrahour dataset date range
# and remove the structurally different high-volatility 2000-2004 regime
data_full = data_full[data_full.index >= '2005-01-03']

# Select and reorder columns into the canonical feature ordering for the modelling pipeline
data_full = data_full[['Return_Squared', 'Return', 'Range', 'Volume', 'VIX', 'Daily Volatility', 'target']]

print(data_full.head(10))
print(f'\nDate range: {data_full.index.min()} to {data_full.index.max()}')
print(f'Total rows: {len(data_full):,}')
print(f'Target distribution: {data_full["target"].value_counts().to_dict()}')

            Return_Squared    Return   Range        Volume    VIX  \
Date                                                                
2005-01-03        0.000001  0.001018  0.0103  1.066716e+09  14.08   
2005-01-04        0.002351 -0.048485  0.0140  7.902240e+08  13.98   
2005-01-05        0.000092  0.009569  0.0070  7.248120e+08  14.09   
2005-01-06        0.000092 -0.009569  0.0088  5.635680e+08  13.58   
2005-01-07        0.000377 -0.019418  0.0082  7.630920e+08  13.49   
2005-01-10        0.000005  0.002176  0.0029  5.795520e+08  13.23   
2005-01-11        0.000990 -0.031468  0.0076  1.093380e+09  13.19   
2005-01-12        0.000071 -0.008448  0.0081  1.057800e+09  12.56   
2005-01-13        0.000114  0.010689  0.0100  8.987640e+08  12.84   
2005-01-14        0.000008  0.002794  0.0049  5.739120e+08  12.43   

            Daily Volatility  target  
Date                                  
2005-01-03          0.012517       1  
2005-01-04          0.011489       0  
2005-01-05     

---
## 9. Export to CSV

The final dataset is exported to `NVDA_Intraday_Volatility_Dataset.csv` with the date as the index.

In [9]:
# Export final dataset with Date as the index
# Save with VIX suffix to distinguish from the original dataset
data_full.to_csv('NVDA_Intraday_Volatility_Dataset_VIX.csv')
print('Saved to NVDA_Intraday_Volatility_Dataset_VIX.csv')
print(f'Columns: {list(data_full.columns)}')
print(f'Shape:   {data_full.shape}')

Saved to NVDA_Intraday_Volatility_Dataset_VIX.csv
Columns: ['Return_Squared', 'Return', 'Range', 'Volume', 'VIX', 'Daily Volatility', 'target']
Shape:   (5323, 7)
